# 06 — Compare Model Serving vs Databricks App

Head-to-head latency probe. Both endpoints run the **identical** `claims_fe`
wheel code — only the serving shell differs:

| | Stack | Runtime |
|---|---|---|
| Pyfunc endpoint (`claims-fe-transformer`) | MLflow scoring server + gunicorn | Model Serving workload_size=Small |
| App (`claims-fe-app`) | FastAPI + uvicorn + orjson | Databricks Apps (2 vCPU / 6 GB) |

**What this notebook measures:**
1. Response equality — both endpoints return the same features for the same input.
2. Latency distribution (p50 / p95 / p99) at concurrency 1 / 10 / 50.
3. Payload-size sensitivity — small / medium / large.

Output: a pandas DataFrame table, printable and comparable.

In [0]:
%pip install --upgrade databricks-sdk requests
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
CATALOG = "fins_genai"
SCHEMA = "classic_ml"
TABLE_NAME = "fe_test_payloads"
ENDPOINT_NAME = "claims-fe-transformer"    # Model Serving pyfunc endpoint
APP_NAME = "claims-fe-app"                  # Databricks App

## Resolve endpoints

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
app = w.apps.get(APP_NAME)
APP_URL = app.url.rstrip("/")

# Sanity: pyfunc endpoint must be READY
ep = w.serving_endpoints.get(ENDPOINT_NAME)
assert ep.state.ready.value == "READY", f"pyfunc endpoint not READY: {ep.state.ready.value}"
print(f"Pyfunc endpoint: {ENDPOINT_NAME}  (READY)")

# Sanity: app must be ACTIVE
assert app.compute_status and "ACTIVE" in str(app.compute_status.state), \
    f"app not active: {app.compute_status.state if app.compute_status else None}"
print(f"App URL:         {APP_URL}")

Pyfunc endpoint: claims-fe-transformer  (READY)
App URL:         https://claims-fe-app-1444828305810485.aws.databricksapps.com


## Load and scale the test payloads

Scaling the text fields linearly (see analysis in the conversation history — regex
cost scales ~0.5 ms per KB of text). Three tiers:

In [0]:
import json

rows = (
    spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")
    .select("payload_id", "payload_json")
    .orderBy("payload_id")
    .collect()
)
base_payloads = [r.payload_json for r in rows]


def scale_payload(payload_json: str, mult: int) -> str:
    p = json.loads(payload_json)
    claim = p["Claims"][0]
    for field in ("ConcatenatedDocs", "ConcatenatedNotes", "ConcatenatedIncidents"):
        orig = claim.get(field) or ""
        claim[field] = " ".join([orig] * mult) if mult > 1 else orig
    return json.dumps(p)


SIZE_TIERS = {
    "small  (~2KB)":    (1,   base_payloads),
    "medium (~40KB)":   (20,  [scale_payload(p, 20)  for p in base_payloads]),
    "large  (~200KB)":  (100, [scale_payload(p, 100) for p in base_payloads]),
}

for name, (mult, payloads) in SIZE_TIERS.items():
    avg_kb = sum(len(p) for p in payloads) / len(payloads) / 1024
    print(f"  {name:18s}  {len(payloads)} payloads, avg {avg_kb:.1f} KB")

  small  (~2KB)       50 payloads, avg 16.0 KB
  medium (~40KB)      50 payloads, avg 302.9 KB
  large  (~200KB)     50 payloads, avg 1510.7 KB


## Callers

In [0]:
import requests

# Databricks Apps require an audience-scoped OAuth JWT, not the notebook's
# internal runtime token.  Exchange once and reuse for the session.
_app_session = requests.Session()

def _get_app_oauth_token() -> str:
    """Exchange the notebook runtime token for an audience-scoped JWT."""
    host = w.config.host.rstrip("/")
    notebook_token = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().apiToken().get()
    )
    app_client_id = w.apps.get(APP_NAME).oauth2_app_client_id
    resp = requests.post(
        f"{host}/oidc/v1/token",
        data={
            "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
            "subject_token": notebook_token,
            "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
            "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
            "scope": "all-apis",
            "audience": app_client_id,
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]

_app_token = _get_app_oauth_token()


def _app_auth_request(payload_json: str) -> requests.PreparedRequest:
    req = requests.Request(
        "POST",
        f"{APP_URL}/transform",
        json={"payload_json": payload_json},
    ).prepare()
    req.headers["Authorization"] = f"Bearer {_app_token}"
    return req


def call_pyfunc(payload_json: str) -> dict:
    resp = w.serving_endpoints.query(
        name=ENDPOINT_NAME,
        dataframe_records=[{"payload_json": payload_json}],
    )
    return json.loads(resp.predictions[0]["features_json"])


def call_app(payload_json: str) -> dict:
    req = _app_auth_request(payload_json)
    resp = _app_session.send(req, timeout=120)
    resp.raise_for_status()
    return json.loads(resp.json()["features_json"])

## Equality check
Both endpoints must return identical features for the same input. If they don't,
the wheel versions are out of sync — stop and reconcile before trusting the latency numbers.

In [0]:
mismatches = []
for i, p in enumerate(base_payloads[:10]):
    a = call_pyfunc(p)
    b = call_app(p)
    if a != b:
        diff = [k for k in set(a) | set(b) if a.get(k) != b.get(k)]
        mismatches.append((i, diff))

if mismatches:
    print("MISMATCHES:")
    for i, diff in mismatches[:5]:
        print(f"  payload[{i}]: keys differ: {diff}")
    raise AssertionError(f"{len(mismatches)}/10 equality failures — wheels diverged")
print("Equality check: 10/10 payloads match between pyfunc and app.")

Equality check: 10/10 payloads match between pyfunc and app.


## Latency probe

In [0]:
import concurrent.futures
import random
import statistics
import time
from typing import Callable, List


def probe(caller: Callable[[str], dict], payloads: List[str], concurrency: int, n_requests: int):
    random.seed(1)

    def one_call(p):
        t0 = time.perf_counter()
        try:
            caller(p)
            return ("ok", time.perf_counter() - t0)
        except Exception as e:
            return (f"err:{type(e).__name__}", time.perf_counter() - t0)

    # Warmup: give the caller a few calls before we start measuring (both endpoints
    # are already provisioned so this is about TCP/keepalive reuse, not cold start)
    for _ in range(3):
        one_call(random.choice(payloads))

    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = [pool.submit(one_call, random.choice(payloads)) for _ in range(n_requests)]
        for fut in concurrent.futures.as_completed(futures):
            results.append(fut.result())

    latencies_ms = sorted(r[1] * 1000 for r in results)
    errors = sum(1 for r in results if not r[0].startswith("ok"))
    n = len(latencies_ms)
    return {
        "p50": latencies_ms[n // 2],
        "p95": latencies_ms[int(n * 0.95)],
        "p99": latencies_ms[int(n * 0.99)],
        "max": max(latencies_ms),
        "errors": errors,
    }


In [0]:
import pandas as pd

CONCURRENCIES = [1, 10, 50]

rows = []
for size_label, (_, payloads) in SIZE_TIERS.items():
    for concurrency in CONCURRENCIES:
        n_requests = 30 if concurrency == 1 else 100
        for system, caller in [("pyfunc", call_pyfunc), ("app", call_app)]:
            print(f"probing {system:>6} @ {size_label:18s} c={concurrency:>2} ...")
            r = probe(caller, payloads, concurrency, n_requests=n_requests)
            rows.append({"system": system, "size": size_label, "concurrency": concurrency, **r})

results_df = pd.DataFrame(rows)
print("\nRaw results:")
print(results_df.to_string(index=False))

probing pyfunc @ small  (~2KB)      c= 1 ...
probing    app @ small  (~2KB)      c= 1 ...
probing pyfunc @ small  (~2KB)      c=10 ...
probing    app @ small  (~2KB)      c=10 ...
probing pyfunc @ small  (~2KB)      c=50 ...
probing    app @ small  (~2KB)      c=50 ...
probing pyfunc @ medium (~40KB)     c= 1 ...
probing    app @ medium (~40KB)     c= 1 ...
probing pyfunc @ medium (~40KB)     c=10 ...
probing    app @ medium (~40KB)     c=10 ...
probing pyfunc @ medium (~40KB)     c=50 ...
probing    app @ medium (~40KB)     c=50 ...
probing pyfunc @ large  (~200KB)    c= 1 ...
probing    app @ large  (~200KB)    c= 1 ...
probing pyfunc @ large  (~200KB)    c=10 ...
probing    app @ large  (~200KB)    c=10 ...
probing pyfunc @ large  (~200KB)    c=50 ...
probing    app @ large  (~200KB)    c=50 ...

Raw results:
system            size  concurrency          p50          p95          p99          max  errors
pyfunc   small  (~2KB)            1    55.453851    83.322480    98.393305    98

## Pivot — pyfunc vs app side by side

In [0]:
pivot = results_df.pivot_table(
    index=["size", "concurrency"],
    columns="system",
    values=["p50", "p95", "p99", "errors"],
    aggfunc="first",
)
# Reorder for readability
pivot = pivot.reindex(columns=pd.MultiIndex.from_product([["p50", "p95", "p99", "errors"], ["pyfunc", "app"]]))
print("Side-by-side (values in ms, errors in count):")
pivot

Side-by-side (values in ms, errors in count):


p50                ... errors    
                                   pyfunc           app  ... pyfunc app
size            concurrency                              ...           
large  (~200KB) 1             1236.064607    824.355286  ...      0   0
                10            6224.599393   8171.922597  ...      0   0
                50           30407.111666  31835.627624  ...      0   0
medium (~40KB)  1              281.631433    176.619267  ...      0   0
                10            1333.088441   1660.134023  ...      0   0
                50            4801.031694   5423.642266  ...      0   0
small  (~2KB)   1               55.453851     21.848491  ...      0   0
                10              64.416339     97.088949  ...      0   0
                50              98.954453    125.191017  ...      0   0

[9 rows x 8 columns]

## Delta view

Positive = app is faster. Negative = pyfunc is faster. Large error rates in any
cell should discredit that row's comparison.

In [0]:
delta_rows = []
for (size, conc), group in results_df.groupby(["size", "concurrency"]):
    row = {"size": size, "concurrency": conc}
    for col in ("p50", "p95", "p99"):
        pyfunc_val = group[group["system"] == "pyfunc"][col].iloc[0]
        app_val = group[group["system"] == "app"][col].iloc[0]
        row[f"{col}_pyfunc_minus_app_ms"] = round(pyfunc_val - app_val, 1)
    delta_rows.append(row)

delta_df = pd.DataFrame(delta_rows)
print("Pyfunc - App latency (ms) — positive means app is faster:")
delta_df

Pyfunc - App latency (ms) — positive means app is faster:


,size,concurrency,p50_pyfunc_minus_app_ms,p95_pyfunc_minus_app_ms,p99_pyfunc_minus_app_ms
0,large (~200KB),1,411.7,522.4,647.3
1,large (~200KB),10,-1947.3,1061.9,2359.5
2,large (~200KB),50,-1428.5,-28918.8,-25221.6
3,medium (~40KB),1,105.0,129.1,144.5
4,medium (~40KB),10,-327.0,-288.6,-9.8
5,medium (~40KB),50,-622.6,-3972.2,-2973.9
6,small (~2KB),1,33.6,34.1,46.1
7,small (~2KB),10,-32.7,-44.8,12.2
8,small (~2KB),50,-26.2,-26.8,43.1


## How to read this

- **At concurrency=1**: the delta is the pure serving-shell overhead — MLflow
  middleware + pandas conversion vs FastAPI + orjson.
- **At concurrency=10/50**: the delta additionally reflects worker / GIL behaviour
  under load. If the app scales better here, it's a real architectural advantage;
  if pyfunc holds up, Model Serving's higher workload size is compensating.
- **Larger size tiers** magnify the body-decode cost on both sides — MLflow's
  pandas-DataFrame pipeline is where we expected the biggest pyfunc tax.
- **Error rates** above 1% in any cell mean the comparison isn't trustworthy for
  that point — usually worker saturation or request timeouts. Use that as a
  signal to re-run with lower concurrency or to bump workload size.